In [1]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [2]:
pip install -U langchain-community sentence-transformers

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_7537/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saravananselvamohan/tamil-kollywood-films-dataset")

print("Path to dataset files:", path)

100%|██████████| 24.6k/24.6k [00:00<00:00, 28.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/saravananselvamohan/tamil-kollywood-films-dataset/versions/2


In [34]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/Tamil_movies_dataset.csv")
documents = loader.load()

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0
)

documents = text_splitter.split_documents(documents)

In [36]:
from langchain_community.vectorstores import Chroma

In [7]:
pip install chromadb

In [37]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents,
    embeddings
)

In [38]:

# create retriever
retriever = vectorstore.as_retriever()

In [39]:
pip install langchain-groq groq

In [40]:
!pip install rank_bm25

from langchain_community.retrievers import BM25Retriever

keyword_retriever = BM25Retriever.from_documents(documents)
keyword_retriever.k = 3

In [42]:
from langchain_classic.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(retrievers=[retriever, keyword_retriever], weights=[0.5, 0.5])

In [46]:
# ── RAG Chain ─────────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

template = """
You are a helpful assistant that answers questions based on the following context.
If you don't find the answer in the context, just say that you don't know.

Context: {context}

Question: {input}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": ensemble_retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke("give a surya films")
print(response)

Surya films are:

1. Singam 3 (2017)
